# Exercise 5 – LU Decomposition (Task 1)

In this notebook you implement the LU decomposition from the lecture yourself and see where it fails.

**Rules for this part: no AI tools.** Write the code yourself and make sure you understand every line.
(In Task 2 on the exercise sheet you may use AI.)

**How it works:** Fill in the gaps marked `# YOUR CODE HERE` and delete the line `raise NotImplementedError`.
After each gap a check prints ✓ (correct) or ✗ (wrong). Run the cells from top to bottom (*Shift+Enter*).
There are only three gaps (sections 1 and 2); section 4 is code to read and run.

> **Save your work!** This Binder session is temporary and is deleted after a few minutes of inactivity.
> Use *File → Download* now and then. You need your functions again for Task 2.

## Recap

The LU decomposition writes a square matrix as a product $A = LU$ of a lower triangular matrix $L$
(with ones on the diagonal) and an upper triangular matrix $U$. Then $A\mathbf x = \mathbf b$ is solved in two steps:

1. $L\mathbf y = \mathbf b$ by **forward substitution**: $\quad y_i = \Big(b_i - \sum_{j<i} l_{ij}\, y_j\Big) / l_{ii}, \quad i = 0, 1, \dots, n-1$
2. $U\mathbf x = \mathbf y$ by **back substitution**: $\quad x_i = \Big(y_i - \sum_{j>i} u_{ij}\, x_j\Big) / u_{ii}, \quad i = n-1, n-2, \dots, 0$

**Indices start at 0 in this notebook, as in Python** (the lecture starts at 1: $a_{11}$ there is $a_{00}$ here).

In [ ]:
import numpy as np
import scipy.linalg

np.set_printoptions(precision=4, suppress=True)


def check(label, ok):
    print(("✓  " if ok else "✗  ") + label)

## 1  Triangular systems

`forward_subst` is given: `L[i, :i] @ y[:i]` computes the sum $\sum_{j<i} l_{ij}\, y_j$ over the values $y_j$
that are already known. Complete `back_subst` in the same way – which values $x_j$ are already known in row $i$?

In [ ]:
def forward_subst(L, b):
    """Solve L y = b for a lower triangular matrix L."""
    n = len(b)
    y = np.zeros(n)
    for i in range(n):                      # i = 0, 1, ..., n-1
        y[i] = (b[i] - L[i, :i] @ y[:i]) / L[i, i]
    return y


def back_subst(U, y):
    """Solve U x = y for an upper triangular matrix U."""
    n = len(y)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):          # i = n-1, n-2, ..., 0
        # YOUR CODE HERE
        raise NotImplementedError
    return x

In [ ]:
# check
rng = np.random.default_rng(1)
T = np.triu(rng.uniform(-1, 1, (6, 6))) + 2 * np.eye(6)     # upper triangular test matrix
y = rng.uniform(-1, 1, 6)
check("back_subst", np.allclose(back_subst(T, y), scipy.linalg.solve_triangular(T, y, lower=False)))

## 2  LU decomposition without pivoting

Gaussian elimination as in the lecture: for every pivot column $k$ and every row $i > k$ below the pivot

$$ l_{ik} = \frac{u_{ik}}{u_{kk}}, \qquad \text{row}_i \leftarrow \text{row}_i - l_{ik}\cdot \text{row}_k . $$

We start with $L = I$ and $U = A$. The factor $l_{ik}$ is written into $L$, the row operation is done in $U$.
After every row operation $A = LU$ is still true; at the end $U$ is upper triangular.

**Example from the lecture.** At the start $A = LU$ holds trivially:

$$
\begin{bmatrix} 1&2&3\\1&1&1\\3&3&1 \end{bmatrix}
= \begin{bmatrix} 1&0&0\\0&1&0\\0&0&1 \end{bmatrix}
  \begin{bmatrix} 1&2&3\\1&1&1\\3&3&1 \end{bmatrix}
$$

**One row operation** (pivot column $k = 0$, row $i = 1$):
1. factor $l_{10} = u_{10} / u_{00} = 1/1 = 1$, written into $L$,
2. row 1 of $U$ minus $l_{10}$ times row 0: $\;(1, 1, 1) - 1\cdot(1, 2, 3) = (0, -1, -2)$,
   element by element $\;u_{1j} \leftarrow u_{1j} - l_{10}\, u_{0j}\;$ for $j = 0, 1, 2$.

$$
A = \begin{bmatrix} 1&0&0\\\mathbf{1}&1&0\\0&0&1 \end{bmatrix}
    \begin{bmatrix} 1&2&3\\\mathbf{0}&\mathbf{-1}&\mathbf{-2}\\3&3&1 \end{bmatrix}
$$

The same for row $i = 2$ ($l_{20} = 3/1 = 3$) finishes column 0.
Then column $k = 1$, row $i = 2$ ($l_{21} = -3/-1 = 3$):

$$
A = \begin{bmatrix} 1&0&0\\1&1&0\\3&0&1 \end{bmatrix}
    \begin{bmatrix} 1&2&3\\0&-1&-2\\0&-3&-8 \end{bmatrix}
  = \begin{bmatrix} 1&0&0\\1&1&0\\3&3&1 \end{bmatrix}
    \begin{bmatrix} 1&2&3\\0&-1&-2\\0&0&-2 \end{bmatrix}
$$

**Warm-up:** the first row operation is given as code. Write the other two (with numbers, no loops over $k$ and $i$ yet).

In [ ]:
A1 = np.array([[1, 2, 3],
               [1, 1, 1],
               [3, 3, 1]])

U = np.array(A1, dtype=float)       # a float copy of A, it becomes U
L = np.eye(3)

# k = 0, i = 1   (given)
L[1, 0] = U[1, 0] / U[0, 0]
for j in range(3):
    U[1, j] -= L[1, 0] * U[0, j]
check("after (k=0, i=1): L U = A", np.allclose(L @ U, A1))

# k = 0, i = 2   and   k = 1, i = 2
# YOUR CODE HERE
raise NotImplementedError

print("L =\n", L, "\nU =\n", U)
check("after all three row operations: L U = A", np.allclose(L @ U, A1))
check("U is upper triangular", np.allclose(U, np.triu(U)))

Now the general version: the same row operation, with the numbers replaced by `k`, `i` (and the column loop over `j`).
Complete the body of the `i` loop. **Use the explicit loop over `j`** as in the warm-up – in Task 2 you will time
exactly this version.

In [ ]:
def lu_nopivot(A):
    """LU decomposition A = L U without pivoting."""
    n = A.shape[0]
    U = np.array(A, dtype=float)     # a float copy of A, it becomes U  (why float?)
    L = np.eye(n)
    for k in range(n - 1):           # pivot column k  (one step of the lecture)
        for i in range(k + 1, n):    # rows below the pivot
            # YOUR CODE HERE
            raise NotImplementedError
    return L, U


def lu_solve_nopivot(L, U, b):
    """Solve L U x = b: first L y = b, then U x = y."""
    y = forward_subst(L, b)
    return back_subst(U, y)

In [ ]:
# check 1: example from the lecture
L, U = lu_nopivot(A1)
check("lecture example: L", np.allclose(L, [[1, 0, 0], [1, 1, 0], [3, 3, 1]]))
check("lecture example: U", np.allclose(U, [[1, 2, 3], [0, -1, -2], [0, 0, -2]]))
x = lu_solve_nopivot(L, U, np.array([2, 2, 0]))
print("x =", x)
check("lecture example: x = (5, -6, 3)", np.allclose(x, [5, -6, 3]))

# check 2: larger matrices
A2 = rng.uniform(-1, 1, (6, 6)) + 6 * np.eye(6)
L, U = lu_nopivot(A2)
check("6x6 matrix: L U = A", np.allclose(L @ U, A2))

A3 = np.array([[ 7,  3, -1,  2],
               [ 3,  8,  1, -4],
               [-1,  1,  4, -1],
               [ 2, -4, -1,  6]])
b3 = np.array([1, 2, 3, 4])
L, U = lu_nopivot(A3)
check("4x4 system, compared with np.linalg.solve", np.allclose(lu_solve_nopivot(L, U, b3), np.linalg.solve(A3, b3)))

## 3  When does it fail?

**Predict first, then run the cells.** Both matrices are invertible, so both systems have a unique solution.

**(a)** The anti-diagonal matrix
$$ J = \begin{bmatrix} 0&0&0&1\\0&0&1&0\\0&1&0&0\\1&0&0&0 \end{bmatrix} $$

In [ ]:
J = np.fliplr(np.eye(4))
L, U = lu_nopivot(J)
print("L =\n", L, "\nU =\n", U)

**(b)** The system
$$ \begin{bmatrix} \varepsilon & 1\\ 1 & 1 \end{bmatrix} \mathbf x = \begin{bmatrix} 1\\ 2 \end{bmatrix}
\qquad \text{with } \varepsilon = 10^{-20}, \qquad \text{exact solution } \mathbf x = \Big(\tfrac{1}{1-\varepsilon},\ \tfrac{1-2\varepsilon}{1-\varepsilon}\Big) \approx (1, 1). $$

In [ ]:
eps = 1e-20
A_eps = np.array([[eps, 1],
                  [1,   1]])
b_eps = np.array([1, 2])
L, U = lu_nopivot(A_eps)
print("L =\n", L, "\nU =\n", U)
print("x =", lu_solve_nopivot(L, U, b_eps), "  (exact: approx. [1, 1])")

**Questions** (discuss with your group):
- What goes wrong in (a)? Which line of your code causes it?
- In (b) nothing crashes, but the result is wrong. Compute $l_{10}$ and $u_{11}$ by hand and compare with the printed
  `L` and `U`. Which information is lost? (Try `1 - 1e20` in Python.)
- Would it help to write the two equations in the other order?

## 4  Gaussian elimination with row swaps (pivoting)

Both problems of section 3 come from a bad pivot $u_{kk}$: zero in (a), tiny in (b) (then $l_{ik}$ is huge).
The fix is simple: **swapping two equations does not change the solution.** So before eliminating column $k$,
swap equation $k$ with the equation $m \ge k$ below it that has the **largest** $|u_{mk}|$.
This is called (partial) **pivoting**. Then the pivot is never zero (for an invertible $A$) and $|l_{ik}| \le 1$.

Since the equations are swapped, the right-hand side $\mathbf b$ has to be swapped as well. We therefore eliminate
on the augmented matrix $[A \mid \mathbf b]$ as in the lecture. For system (b):

$$
\left[\begin{array}{cc|c} \varepsilon & 1 & 1\\ 1 & 1 & 2\end{array}\right]
\xrightarrow{\text{swap rows 0 and 1}}
\left[\begin{array}{cc|c} 1 & 1 & 2\\ \varepsilon & 1 & 1\end{array}\right]
\xrightarrow{\text{row}_1 - \varepsilon\cdot\text{row}_0}
\left[\begin{array}{cc|c} 1 & 1 & 2\\ 0 & 1-\varepsilon & 1-2\varepsilon\end{array}\right]
$$

Now the factor is $\varepsilon$ instead of $10^{20}$ and nothing is lost: back substitution gives $\mathbf x \approx (1, 1)$.

**Swapping rows in NumPy.** The obvious Python line does *not* work for NumPy arrays. Run the cell and look at the result:

In [ ]:
M = np.array([[1, 2],
              [3, 4]])
M[0], M[1] = M[1], M[0]      # wrong for NumPy arrays: M[0] and M[1] are views, not copies
print(M)

M = np.array([[1, 2],
              [3, 4]])
M[[0, 1]] = M[[1, 0]]        # correct: fancy indexing makes a copy of the rows first
print(M)

The function below is complete. **Read it and compare it with `lu_nopivot`:** only the marked lines are new, and the
right-hand side `c` is eliminated together with `U`. The factors $l_{ik}$ are not stored here (we solve directly).

*Bonus (if you have time):* delete the three marked lines and write them yourself.
Tip: `np.argmax(np.abs(U[k:, k]))` gives the position of the largest entry *relative to row k*.

In [ ]:
def gauss_solve_pivot(A, b):
    """Solve A x = b by Gaussian elimination with row swaps (partial pivoting)."""
    n = len(b)
    U = np.array(A, dtype=float)                # becomes the upper triangular matrix
    c = np.array(b, dtype=float)                # right-hand side, swapped and eliminated together with U
    for k in range(n - 1):
        # --- new: pivot search and row swap ---------------------------------------------------
        m = k + np.argmax(np.abs(U[k:, k]))     # row m >= k with the largest |U[m, k]|
        U[[k, m]] = U[[m, k]]                   # swap equations k and m ...
        c[[k, m]] = c[[m, k]]                   # ... including the right-hand side
        # --- elimination below the pivot, as in lu_nopivot ------------------------------------
        for i in range(k + 1, n):
            l = U[i, k] / U[k, k]
            for j in range(k, n):
                U[i, j] -= l * U[k, j]
            c[i] -= l * c[k]
    return back_subst(U, c)

In [ ]:
# check: the two problem cases from section 3, and a random matrix
x = gauss_solve_pivot(J, np.array([1, 2, 3, 4]))
print("x =", x)
check("anti-diagonal matrix: x = (4, 3, 2, 1)", np.allclose(x, [4, 3, 2, 1]))

x = gauss_solve_pivot(A_eps, b_eps)
print("x =", x)
check("epsilon system: x = (1, 1)", np.allclose(x, [1, 1], rtol=1e-14))

B = rng.uniform(-1, 1, (8, 8))
c = rng.uniform(-1, 1, 8)
check("random 8x8 matrix, compared with np.linalg.solve", np.allclose(gauss_solve_pivot(B, c), np.linalg.solve(B, c)))

**Questions:**
- Why does `c` have to be swapped together with `U`?
- Why is the row with the *largest* $|u_{mk}|$ chosen, and not just any row with $u_{mk} \neq 0$? (Look at $l_{ik}$.)

**Outlook.** If one also stores the factors $l_{ik}$ and remembers the row swaps in a permutation matrix $P$,
one gets the LU decomposition with pivoting, $PA = LU$. This is what `scipy.linalg.lu_factor` computes; you use it in Task 2.

## 5  Save your functions for Task 2

The next cell writes your functions into the file `my_lu.py`.
Download it (file browser on the left → right click → *Download*), together with this notebook.
In Task 2 you can then use them with `from my_lu import *`.

In [ ]:
import inspect

functions = [forward_subst, back_subst, lu_nopivot, lu_solve_nopivot, gauss_solve_pivot]
with open("my_lu.py", "w") as f:
    f.write('"""LU decomposition, exercise 5, task 1."""\nimport numpy as np\n\n\n')
    f.write("\n\n".join(inspect.getsource(fn) for fn in functions))
print("written: my_lu.py")